# News data workflow

In [1]:
# Import required libraries
import newsapi
import dlt
import duckdb
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
from dotenv import load_dotenv
import time
from newspaper import Article
from dateutil import parser

def to_naive_iso(dt):
    if dt is None:
        return None
    # parse strings or accept datetimes
    dt = parser.isoparse(dt) if isinstance(dt, str) else dt
    if getattr(dt, "tzinfo", None):
        dt = dt.astimezone(tz=None)  # convert to local tz if desired
        dt = dt.replace(tzinfo=None)
    return dt.isoformat()

# Load environment variables
load_dotenv()

print("✅ All libraries imported successfully!")
print(f"Current working directory: {os.getcwd()}")


✅ All libraries imported successfully!
Current working directory: /Users/marcogaudio/github/data-engineering-environment-news/notebooks


In [ ]:
pipeline = dlt.pipeline(
    # how the pipeline will be named in the DLT UI
    pipeline_name="news_full_content_data",
    # where the data will be stored, in this case we are using DuckDB
    destination="duckdb",
    # the name of the schema where the data will be stored in DuckDB
    dataset_name="news_data"
)

print("✅ DLT pipeline initialized successfully!")
print(f"Pipeline name: {pipeline.pipeline_name}")
print(f"Destination: {pipeline.destination}")
print(f"Dataset: {pipeline.dataset_name}")

✅ DLT pipeline initialized successfully!
Pipeline name: news_full_content_pipeline
Destination: <dlt.destinations.duckdb(destination_type='duckdb', staging_dataset_name_layout='%s_staging', enable_dataset_name_normalization=True, info_tables_query_threshold=1000, truncate_tables_on_staging_destination_before_load=True, local_dir='/Users/marcogaudio/github/data-engineering-environment-news/notebooks', pipeline_name='news_full_content_pipeline', pipeline_working_dir='/Users/marcogaudio/.dlt/pipelines/news_full_content_pipeline', create_indexes=False)>
Dataset: news_data


In [7]:
@dlt.resource(write_disposition="append")
def news_with_full_content(days_back: int = 2):    
    # loading api key from .env
    api_key = os.getenv("NEWS_API_KEY")
    if not api_key:
        # handling missing API key scenario
        raise ValueError("NEWS_API_KEY not found in environment variables.")
    
    api_client = newsapi.NewsApiClient(api_key=api_key)

    # testing API connection 
    try:
        sources = api_client.get_sources()
        print(f"✅ Connesso! Fonti disponibili: {len(sources['sources'])}")
    except Exception as e:
        print(f"❌ Errore durante la connessione all'API: {e}")

    today = datetime.now()
    limit = today - timedelta(days=days_back)
    # Fetching news data  
    try:
        
        all_articles = api_client.get_everything(
            q="Artificial Intelligence",
            from_param=limit.strftime('%Y-%m-%d'),
            to=today.strftime("%Y-%m-%d"),
            language="en",
            sort_by="relevancy",
            page_size=100
        )

        print(f"✅ Dati recuperati! Numero di articoli: {len(all_articles['articles'])}")
    
    except Exception as e:
        print(f"❌ Errore durante il recupero dei dati: {e}")

    for art in all_articles['articles']:
        try:
            # Scrape full content
            article = Article(art['url'])
            article.download()
            article.parse()

            if len(article.text.split()) < 200:  # Skip articles with very short content
                print(f"⚠️ Skip {art['url']}: content too short")
                continue

            yield {
                'title': art['title'],
                'source': art['source']['name'],
                'author': art['author'],
                'published_at': art['publishedAt'],
                'url': art['url'],
                'description': art['description'],
                'full_text': article.text,
                'word_count': len(article.text.split()),
                'top_image': article.top_image,
                'extracted_at': datetime.now().isoformat()
            }
            
            time.sleep(1)
            
        except Exception as e:
            print(f"⚠️ Skip {art['url']}: {e}")
            continue

In [8]:
# Carica
load_info = pipeline.run(
    news_with_full_content(),
    table_name="news_table",
    write_disposition="replace"
)

print(f"loaded info: {load_info}")
# Get the database path and connect directly to DuckDB
db_path = pipeline.sql_client().credentials.database
print(f"\n🔗 Database path: {db_path}")

✅ Connesso! Fonti disponibili: 125
✅ Dati recuperati! Numero di articoli: 96
⚠️ Skip https://www.npr.org/2026/05/29/g-s1-124599/tennis-biden-bezos-pope-google-news-quiz: content too short
⚠️ Skip https://gizmodo.com/star-wars-and-jurassic-world-director-gareth-edwards-is-all-for-ai-in-filmmaking-2000765061: Article `download()` failed with Website protected with Cloudflare, url: None on URL https://gizmodo.com/star-wars-and-jurassic-world-director-gareth-edwards-is-all-for-ai-in-filmmaking-2000765061
⚠️ Skip https://www.golem.de/sonstiges/zustimmung/auswahl.html?from=https%3A%2F%2Fwww.golem.de%2Fnews%2Fartificial-intelligence-anthropic-finalises-65bn-funding-deal-to-surpass-openai-s-valuation-2605-209189.html&referer=https%3A%2F%2Ft.co%2F0039202b33: content too short
⚠️ Skip https://www.abc.net.au/news/2026-05-31/schools-in-asia-embracing-ai/106703054: Article `download()` failed with Status code 403 for url None on URL https://www.abc.net.au/news/2026-05-31/schools-in-asia-embracing-a

In [9]:
# Query DuckDB
con = duckdb.connect(db_path)

# Get all tables from all schemas
all_tables = con.execute("""
    SELECT * 
    FROM news_data.news_table
""").fetchdf()

print(f"\n📋 Available tables:")
print(all_tables)

# ✅ RIMUOVI TIMEZONE dalle colonne datetime
datetime_columns = all_tables.select_dtypes(include=['datetime64[ns, UTC]', 'datetime']).columns

for col in datetime_columns:
    if all_tables[col].dt.tz is not None:  # Se ha timezone
        all_tables[col] = all_tables[col].dt.tz_localize(None)  # Rimuovi timezone

all_tables.to_excel("../data/news_data.xlsx", index=False)


📋 Available tables:
                                                title                 source  \
0   We Asked the ‘Future of Truth’ Author to Expla...                  Wired   
1   Dell Stock Surges 32% in One Day. Big Revenue ...           Slashdot.org   
2   Anthropic surpasses OpenAI to become most valu...          Qazinform.com   
3   AI-Generated Film About Iranian Protest Violen...                   CNET   
4   How the Pope’s Magnifica Humanitas offers a te...  MIT Technology Review   
..                                                ...                    ...   
62  Zeta Global (ZETA) Soars 25% as CEO ‘Highly Op...    Yahoo Entertainment   
63  WhiteFiber, Inc. (WYFI): Leopold Aschenbrenner...    Yahoo Entertainment   
64  Spatiotemporal changes in inter-city sustainab...             Nature.com   
65  Palantir's 'unlimited access' to patient data ...              TechRadar   
66  AI giant Anthropic reaches near-trillion dolla...    Hurriyet Daily News   

                  

/var/folders/rx/w8wxd2bd5rv_dyjb4h8xq6300000gn/T/ipykernel_85528/416774241.py:20: UserWarning: Cell contents too long (39066), truncated to 32767 characters
  all_tables.to_excel("../data/news_data.xlsx", index=False)
